# PyHealth KeyClass Tutorial — Weak Supervision for Clinical Notes

## Goal
Build a small text-classification pipeline from synthetic notes and category
descriptions, without manually labeled training examples. We will mine phrases,
create labeling functions, aggregate their votes, and train a multilabel model.

The example runs offline after installing dependencies. Its lexical mock encoder
is for testing the pipeline, not for assessing clinical accuracy.

**Paper:** Chufan Gao, Mononito Goswami, Jieshi Chen, and Artur Dubrawski,
[Classifying Unstructured Clinical Notes via Automatic Weak Supervision](https://arxiv.org/abs/2206.12088v2).

## Setup

Use Python 3.12 or 3.13 and a notebook kernel from the environment where this
checkout is installed. From the repository root, install the local implementation:

```bash
pip install -e '.[keyclass]'
pip install ipykernel
```

The `keyclass` extra installs Snorkel. The majority-vote baseline does not need it.
No pretrained model or patient data is downloaded by this notebook.

In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import HashingVectorizer

from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.models import KeyClass
from pyhealth.weak_supervision import (
    KeywordLabeler, aggregate_votes, vote_diagnostics,
)

torch.manual_seed(0)
aggregation = "majority"  # Change to "snorkel" to train with Snorkel targets.

## Create synthetic notes and category descriptions

Each row carries a patient ID and text. Real inputs can also carry an admission
ID (`hadm_id`). Gold diagnosis labels are kept separately for evaluation and are
never passed into phrase mining or weak-label aggregation.

The insertion order of `descriptions` defines the output category order. This
example has three categories; the same interface accepts the agreed 19 ICD-9
categories. Split real data by patient before fitting any component.

In [3]:
descriptions = {
    "respiratory": "pneumonia asthma lung",
    "circulatory": "heart cardiac hypertension",
    "renal": "kidney renal dialysis",
}
# Teammate boundary: rows need text; patient/admission IDs stay with rows.
# Gold labels are deliberately absent from all weak-supervision operations.
rows = [
    {"patient_id": str(i), "text": text}
    for i, text in enumerate(
        [
            "pneumonia lung",
            "asthma lung",
            "heart cardiac",
            "heart hypertension",
            "kidney renal",
            "renal dialysis",
            "pneumonia heart",
            "kidney heart",
            "routine observation",
        ]
    )
]
texts = [r["text"] for r in rows]

pd.DataFrame(rows)

,patient_id,text
0,0,pneumonia lung
1,1,asthma lung
2,2,heart cardiac
3,3,heart hypertension
4,4,kidney renal
5,5,renal dialysis
6,6,pneumonia heart
7,7,kidney heart
8,8,routine observation


## Provide an encoder

The encoder interface is `encode(texts) -> numpy array [number of texts, dimension]`.
Here, a hashing vectorizer supplies deterministic lexical vectors. A clinical
transformer can later replace it without changing the labeler or classifier.

In [4]:
class SyntheticEncoder:
    """Offline lexical mock, NOT a clinical semantic encoder."""

    def encode(self, texts):
        return (
            HashingVectorizer(n_features=64, alternate_sign=False)
            .transform(texts)
            .toarray()
        )


encoder = SyntheticEncoder()

## Generate labeling functions

`KeywordLabeler` extracts frequent 1–3 grams from training notes, filters English
stopwords, and assigns each phrase to its nearest category description using
cosine similarity. It keeps up to `top_k` phrases per category. `min_df=1` keeps
phrases occurring in at least one synthetic note.

Every retained phrase becomes a rule: vote for its category when the phrase
matches; otherwise abstain. Mining and matching use the same normalization.

In [5]:
labeler = KeywordLabeler(encoder, top_k=3, min_df=1).fit(texts, descriptions)
pd.DataFrame([
    {"phrase": lf.phrase,
     "category": labeler.category_ids_[lf.category_index],
     "similarity": round(lf.similarity, 3)}
    for lf in labeler.lfs_
])

,phrase,category,similarity
0,asthma lung,respiratory,0.816
1,kidney heart,respiratory,0.816
2,pneumonia lung,respiratory,0.816
3,heart cardiac,circulatory,0.816
4,heart hypertension,circulatory,0.816
5,cardiac,circulatory,0.577
6,kidney renal,renal,0.816
7,renal dialysis,renal,0.816
8,dialysis,renal,0.577


## Inspect votes and coverage

The vote matrix has one row per note and one column per labeling function.
`-1` means abstain; other values index `labeler.category_ids_`. A note may receive
votes for several categories. Such disagreement can reflect true comorbidity.

In [6]:
votes = labeler.transform(texts)
print("Category order:", labeler.category_ids_)
diagnostics = vote_diagnostics(votes, len(descriptions))
print("Coverage:", round(diagnostics["coverage"], 3))
print("Cross-category fraction:", round(diagnostics["cross_category_fraction"], 3))
pd.DataFrame(votes, columns=[lf.phrase for lf in labeler.lfs_])

Category order: ('respiratory', 'circulatory', 'renal')
Coverage: 0.778
Cross-category fraction: 0.0


,asthma lung,kidney heart,pneumonia lung,heart cardiac,heart hypertension,cardiac,kidney renal,renal dialysis,dialysis
0,-1,-1,0,-1,-1,-1,-1,-1,-1
1,0,-1,-1,-1,-1,-1,-1,-1,-1
2,-1,-1,-1,1,-1,1,-1,-1,-1
3,-1,-1,-1,-1,1,-1,-1,-1,-1
4,-1,-1,-1,-1,-1,-1,2,-1,-1
5,-1,-1,-1,-1,-1,-1,-1,2,2
6,-1,-1,-1,-1,-1,-1,-1,-1,-1
7,-1,0,-1,-1,-1,-1,-1,-1,-1
8,-1,-1,-1,-1,-1,-1,-1,-1,-1


## Aggregate weak labels

The majority baseline normalizes category vote counts, preserving ties. Snorkel
fits a categorical label model from LF agreements and disagreements; it requires
at least three labeling functions. Change `aggregation` in setup and rerun to
use Snorkel for training.

Both return a categorical distribution that sums to one, not independent disease
probabilities. Notes with no matching rules receive uniform placeholders and
are excluded from initial training using the returned `covered` mask.

In [7]:
targets, covered = aggregate_votes(
    votes, len(descriptions), method=aggregation, seed=0,
)
print(f"Covered training notes: {covered.sum()} / {len(covered)}")
pd.DataFrame(targets, columns=labeler.category_ids_).assign(covered=covered)

Covered training notes: 7 / 9


,respiratory,circulatory,renal,covered
0,1.000000,0.000000,0.000000,True
1,1.000000,0.000000,0.000000,True
2,0.000000,1.000000,0.000000,True
3,0.000000,1.000000,0.000000,True
4,0.000000,0.000000,1.000000,True
5,0.000000,0.000000,1.000000,True
6,0.333333,0.333333,0.333333,False
7,1.000000,0.000000,0.000000,True
8,0.333333,0.333333,0.333333,False


### Compare with Snorkel

This optional cell computes Snorkel targets independently of the selected training
method. A missing installation is reported explicitly; it does not silently
substitute majority voting.

In [8]:
import importlib.util

if importlib.util.find_spec("snorkel") is not None:
    snorkel_targets, snorkel_covered = aggregate_votes(
        votes, len(descriptions), method="snorkel", seed=0,
        n_epochs=100, progress_bar=False,
    )
    print("Snorkel target shape:", snorkel_targets.shape)
    print("Mean absolute difference from majority:", float(np.abs(
        snorkel_targets - aggregate_votes(votes, len(descriptions))[0]
    ).mean()))
else:
    print("Snorkel comparison skipped. Install the keyclass extra to run it.")

Snorkel target shape: (9, 3)
Mean absolute difference from majority: 0.1551819145679474


## Create a PyHealth dataset

Cache document embeddings and pair them with the soft targets, preserving row
order and patient IDs. Both fields use the `tensor` processor. A token-based
multilabel processor would incorrectly treat probability values as category names.

In [9]:
embeddings = encoder.encode(texts)
samples = [
    {
        "patient_id": rows[i]["patient_id"],
        "embedding": embeddings[i].tolist(),
        "weak_labels": targets[i].tolist(),
    }
    for i in np.flatnonzero(covered)
]
dataset = create_sample_dataset(
    samples=samples,
    input_schema={"embedding": "tensor"},
    output_schema={"weak_labels": "tensor"},
)

print("Training samples:", len(dataset))

Training samples: 7


## Train the multilabel classifier

`KeyClass` extends `BaseModel` and returns PyHealth's `loss`, `logit`, `y_prob`,
and `y_true` fields. Its default MLP has four linear layers with LeakyReLU and
dropout between hidden layers. Binary cross-entropy with logits accepts soft
targets. The final sigmoid scores are independent across categories.

We use a short explicit optimization loop to show how the model consumes a
PyHealth batch. Two epochs are only a smoke test, not a convergence experiment.

In [10]:
model = KeyClass(dataset, embeddings.shape[1], labeler.category_ids_)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0015)
loader = get_dataloader(dataset, batch_size=4, shuffle=True)
for _ in range(2):
    model.train()
    for batch in loader:
        optimizer.zero_grad()
        result = model(**batch)
        result["loss"].backward()
        optimizer.step()

print("Final batch loss:", float(result["loss"].detach()))

Final batch loss: 0.6759192943572998


## Predict on a new note

Inference needs only document embeddings. These scores show that the pipeline
runs; they are not clinical accuracy estimates. For evaluation, use held-out gold
multi-hot targets in the same category order and document the prediction threshold.
Do not compute classification metrics against the soft training targets.

In [11]:
model.eval()
with torch.no_grad():
    predictions = model(embedding=encoder.encode(["pneumonia heart"]))["y_prob"]
pd.DataFrame(predictions.numpy(), columns=labeler.category_ids_)

,respiratory,circulatory,renal
0,0.500982,0.510068,0.470453


## Checks

Check category alignment, finite normalized targets, the all-abstain case, and
the classifier output shape before connecting real inputs.

In [12]:
assert targets.shape == (len(rows), len(descriptions))
assert np.isfinite(targets).all()
np.testing.assert_allclose(targets.sum(axis=1), 1, atol=1e-6)
assert predictions.shape == (1, len(descriptions))
assert torch.isfinite(predictions).all()
empty_votes = labeler.transform([""])
_, empty_covered = aggregate_votes(empty_votes, len(descriptions))
assert not empty_covered.any()
print("All pipeline checks passed.")

All pipeline checks passed.


## Connect real clinical inputs

Replace `rows` with the training notes and `descriptions` with the ordered category
mapping. Use `ClinicalTextEncoder(model_name=chosen_checkpoint)` in place of the
synthetic encoder, then regenerate rules, targets, and document embeddings.
The clinical adapter uses frozen transformer states, masked mean pooling, and
configurable truncation (512 tokens by default). Record the chosen checkpoint
and text-length policy. This tutorial does not download or evaluate that encoder.

Preserve patient/admission IDs, category order, and LF definitions with the
experiment. Fit mining and aggregation only on the training split. Keep the gold
test labels out of training and model selection.

## Next steps and limitations

This tutorial demonstrates the implemented pipeline, not a completed numerical
reproduction. The exact clinical checkpoint, TF-IDF text selection, confidence
selection, and encoder/self-training schedule remain to be matched to the paper.
`min_df` uses scikit-learn's inclusive threshold. Lexical rules do not resolve
negation, and correlated phrases can overcount evidence.

For the n-gram ablation, compare `ngram_range=(1, 1)` and `(1, 3)` with the same
patient split, encoder, per-class rule budget, seeds, and classifier settings.